<a href="https://colab.research.google.com/github/tiendrebeogoarlette97-bit/NLP-BERT-Sentiment-Classification/blob/main/notebooks/NLP_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification de sentiments : TF-IDF et BERT

**Objectif :** comparer une baseline TF-IDF + régression logistique à BERT-base-uncased fine-tuné sur SST-2 (GLUE). Les modèles seront évalués sur la même partition officielle de validation, avec accuracy, F1 et matrices de confusion. Les labels du test officiel ne sont pas publics. Aucune performance n'est présupposée.

## 1. Installation et configuration

Activer un GPU dans Colab : Exécution > Modifier le type d'exécution > T4 GPU. La seed fixe les générateurs aléatoires, sans garantir une reproductibilité bit à bit sur GPU.

In [ ]:
%pip install -q transformers==5.24.1 datasets==5.0.1 accelerate==1.16.0 pyarrow==23.0.1 jedi

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from datasets import load_dataset
from transformers import set_seed
from IPython.display import display

SEED = 42
set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch :', torch.__version__)
print('Device :', DEVICE)
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
else:
    print('Attention : activer un GPU avant le fine-tuning.')

## 2. Chargement de SST-2

Nous conservons les partitions officielles. Le train sert à apprendre les paramètres et le vocabulaire TF-IDF ; la validation sert à comparer les modèles. Le test est inspecté uniquement pour sa structure.

In [ ]:
dataset = load_dataset('nyu-mll/glue', 'sst2')
print(dataset)
train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()
label_names = {0: 'Negative', 1: 'Positive'}
display(train_df.head(10))
print('Labels du test :', test_df['label'].unique())

## 3. Analyse exploratoire des données

Nous vérifions la qualité des textes, les labels, les doublons et les longueurs. Les doublons sont recensés sans suppression automatique afin de préserver le protocole officiel ; les éventuels chevauchements train/validation seront signalés comme limite.

In [ ]:
summary = []
for name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    normalized = df['sentence'].fillna('').str.strip().str.lower()
    summary.append({'split': name, 'rows': len(df), 'missing_text': df['sentence'].isna().sum(), 'empty_text': normalized.eq('').sum(), 'duplicate_texts': normalized.duplicated().sum(), 'label_values': sorted(df['label'].unique().tolist())})
display(pd.DataFrame(summary))
assert train_df['label'].isin([0, 1]).all()
assert val_df['label'].isin([0, 1]).all()
train_texts = set(train_df['sentence'].str.strip().str.lower())
val_texts = set(val_df['sentence'].str.strip().str.lower())
print('Textes distincts communs train/validation :', len(train_texts & val_texts))

In [ ]:
eda = pd.concat([train_df.assign(split='train'), val_df.assign(split='validation')], ignore_index=True)
eda['sentiment'] = eda['label'].map(label_names)
eda['word_count'] = eda['sentence'].str.split().str.len()
counts = pd.crosstab(eda['split'], eda['sentiment'])
display(counts)
display(pd.crosstab(eda['split'], eda['sentiment'], normalize='index').mul(100).round(2))
display(eda.groupby('split')['word_count'].describe(percentiles=[.5, .9, .95, .99]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, split in zip(axes, ['train', 'validation']):
    sns.countplot(data=eda[eda['split'] == split], x='sentiment', order=['Negative', 'Positive'], ax=ax)
    ax.set_title(f'Distribution des classes — {split}')
    ax.set_xlabel('Sentiment')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
sns.histplot(data=eda, x='word_count', hue='split', stat='density', common_norm=False, element='step', bins=40)
plt.xlabel('Nombre de mots par phrase')
plt.title('Distribution des longueurs de phrases')
plt.tight_layout()
plt.show()

### Interprétation de l'EDA

Résultats observés dans Colab : 67 349 exemples train, 872 validation et 1 821 test. Aucun texte manquant ou vide. Le train contient 371 occurrences redondantes après normalisation (environ 0,55 %), contre aucune en validation et test. Aucun chevauchement textuel exact normalisé train/validation n'a été détecté ; cela n'exclut pas les quasi-doublons. Les classes positives représentent 55,78 % du train et 50,92 % de la validation. Nous conservons les partitions officielles sans rééquilibrage. Les textes de validation sont plus longs (19,55 mots en moyenne contre 9,41), ce qui motive une analyse ultérieure des performances par longueur. Les labels test (-1) ne sont pas publics. Les résultats rapportés concernent la validation officielle.

In [ ]:
duplicate_check = train_df.assign(normalized=train_df['sentence'].str.strip().str.lower()).groupby('normalized')['label'].agg(['size', 'nunique'])
print('Groupes de textes répétés :', int((duplicate_check['size'] > 1).sum()))
print('Groupes avec labels contradictoires :', int((duplicate_check['nunique'] > 1).sum()))

## 4. Baseline : TF-IDF + régression logistique

Le vectoriseur apprend son vocabulaire et ses poids IDF exclusivement sur le train. Nous conservons les négations, utilisons des unigrams et bigrams, et n'appliquons ni suppression de stopwords ni stemming. La régression logistique utilise une régularisation L2 (C=1) et une configuration fixe, sans optimisation sur la validation officielle. Le pipeline empêche la réestimation du vocabulaire sur la validation.

In [ ]:
from time import perf_counter
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay, log_loss
from sklearn.dummy import DummyClassifier

X_train = train_df['sentence'].tolist()
y_train = train_df['label'].to_numpy()
X_val = val_df['sentence'].tolist()
y_val = val_df['label'].to_numpy()

baseline = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, sublinear_tf=True, norm='l2', token_pattern=r'(?u)\b\w+\b')),
    ('lr', LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=SEED))
])
t0 = perf_counter()
baseline.fit(X_train, y_train)
baseline_train_time = perf_counter() - t0
t0 = perf_counter()
baseline_proba = baseline.predict_proba(X_val)
baseline_pred = baseline.classes_[baseline_proba.argmax(axis=1)]
baseline_inference_time = perf_counter() - t0
print('Taille du vocabulaire :', len(baseline.named_steps['tfidf'].vocabulary_))
print('Iterations LR :', baseline.named_steps['lr'].n_iter_)
print('Temps entrainement (s) :', round(baseline_train_time, 3))

In [ ]:
def classification_metrics(y_true, y_pred):
    return {'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0), 'recall': recall_score(y_true, y_pred, zero_division=0), 'f1': f1_score(y_true, y_pred, zero_division=0), 'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0)}

baseline_metrics = classification_metrics(y_val, baseline_pred)
baseline_metrics.update({'train_time_s': baseline_train_time, 'inference_time_s': baseline_inference_time, 'validation_log_loss': log_loss(y_val, baseline_proba, labels=[0,1])})
results = pd.DataFrame([{'model': 'TF-IDF + LR', **baseline_metrics}])
display(results.round(4))
print(classification_report(y_val, baseline_pred, target_names=['Negative', 'Positive'], digits=4))
print('Accuracy train :', accuracy_score(y_train, baseline.predict(X_train)))
dummy = DummyClassifier(strategy='most_frequent').fit(np.zeros((len(y_train), 1)), y_train)
print('Accuracy classe majoritaire :', dummy.score(np.zeros((len(y_val), 1)), y_val))
ConfusionMatrixDisplay.from_predictions(y_val, baseline_pred, display_labels=['Negative', 'Positive'], cmap='Blues')
plt.title('Matrice de confusion — TF-IDF + LR')
plt.tight_layout()
plt.show()

### Interprétabilité de la baseline

Les coefficients positifs et négatifs sont des associations apprises avec la classe positive et négative respectivement. Ils ne prouvent pas une relation causale.

In [ ]:
features = np.array(baseline.named_steps['tfidf'].get_feature_names_out())
weights = baseline.named_steps['lr'].coef_[0]
order = np.argsort(weights)
selected = np.r_[order[:15], order[-15:]]
coef_df = pd.DataFrame({'term': features[selected], 'coefficient': weights[selected]}).sort_values('coefficient')
plt.figure(figsize=(9, 10))
plt.barh(coef_df['term'], coef_df['coefficient'], color=np.where(coef_df['coefficient'] > 0, 'seagreen', 'indianred'))
plt.axvline(0, color='black', linewidth=.8)
plt.title('Termes les plus associés aux sentiments — régression logistique')
plt.xlabel('Coefficient appris')
plt.tight_layout()
plt.show()
val_predictions = val_df[['idx', 'sentence', 'label']].copy()
val_predictions['baseline_pred'] = baseline_pred
val_predictions['baseline_prob_positive'] = baseline_proba[:, 1]
display(val_predictions[val_predictions['label'] != val_predictions['baseline_pred']].head(10))